# Robust Statistics Redo — Results Presentation

**Read-only presentation** of the artifacts stored under `results/exp_r216/` (generated by `031_exp_r216_robust_stats.py`; this notebook makes no computational changes itself).

## Background and Protocol

- **Motivation**: concerns were raised about how the original statistical pipeline (Wilcoxon + Holm) handles the small-sample (16-region) and multi-seed structure, and about the indefensibility of taking the median p-value across three seeds.
- **Inference unit** = region (n=16). Fold-level variation is absorbed into the region×seed unit (the full argument is in the `design_statement` field of `mixed_model.json`).
- **Primary test**: seed averaging → 16 paired regional differences → **exact sign-flip permutation test** (2^16 = 65,536 exhaustive enumeration, two-sided p, minimum attainable p = 2/2^16 ≈ 3.05×10⁻⁵). Static-vs-static comparisons (#5/#6/#7) have no seed dimension and go through the no-seed branch, taking the regional difference directly.
- **CI**: region-level paired bootstrap (outer resampling only, B=10⁴, fixed seed) — used only to report intervals; **no bootstrap p-value is produced**.
- **Per-seed transparency**: for the 6 comparisons involving the GNN, the per-seed permutation p-values are all reported as-is, with no combining (the Cauchy-combined column is for reference only, not the primary figure; **the median is no longer used**, directly addressing the concern about median p-values).
- **Robustness branch**: mixedlm (region + seed **crossed** random intercepts).
- **Multiple-comparison correction**: Holm, **one family per metric** (family = the 9 comparisons in tab:significance, matching the structure of Table 4 in the paper).
- **Spatial dependence**: Moran's I diagnostic on the paired differences (primary W = queen adjacency, supplementary W = centroid kNN(3); 999 permutations, fixed seed); a queen two-sided p<0.05 triggers `spatial_warning` and a spatial-block permutation replacement p (sign flips restricted to connected blocks under queen adjacency).

## Gap in Frozen Artifacts (disclosed as found)

The frozen `static_allocation/all_regions_*.csv` is **missing the GPMpostP arm** (`voronoi_prox2_gpm`, referenced in the paper's Table 3/4); it has been recomputed following the exact same definitional pipeline as script 003, and is backed by strong anchoring against the 3 frozen arms (consistent per-region within storage precision) plus a weak anchor against the paper's Table 3 (±0.005); see `static_arm_recompute_anchor.json`.

In [ ]:
# Read-only: load artifacts
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

OUT = Path('..') / 'results' / 'exp_r216'
registry = pd.read_csv(OUT / 'comparison_registry.csv')
robust = pd.read_csv(OUT / 'robust_tests.csv')
per_seed = pd.read_csv(OUT / 'per_seed_pvalues.csv')
old_vs_new = pd.read_csv(OUT / 'old_vs_new_pvalues.csv')
with open(OUT / 'mixed_model.json', encoding='utf-8') as f:
    mixed = json.load(f)
with open(OUT / 'morans_i.json', encoding='utf-8') as f:
    morans = json.load(f)
with open(OUT / 'static_arm_recompute_anchor.json', encoding='utf-8') as f:
    anchor = json.load(f)
print(f'Comparison × metric cells: {len(robust)} (9 comparisons × 3 metrics)')

## 1. Old vs. New p-Value Comparison Table

The old values come from three sources, shown side by side (not mixed):
- `old_holm_p_paper` / `old_raw_p_paper`: **transcribed verbatim from the paper's tables/text** (the reference baseline);
- `old_*_recomputed`: deterministically recomputed under the old protocol (seed-averaged Wilcoxon + Holm, family=9/metric) — filling in the MAE cells and part of the Corr cells that were never printed in the paper;
- `old_frozen_*`: the repository's frozen `exp2_significance` artifacts (the earlier run in script 008 only covered 7 comparisons, family=7).

**Verification result**: the Wilcoxon p-values recomputed under the old protocol match the frozen artifacts value-for-value within storage precision (bound by the pytest `TestOldProtocolParity` test); the Holm p-values in the paper's Table 4 are confirmed to **mix runs from two different families** (the 7 values from the 7-comparison family plus the later #6/#7 reruns — see the 2.1e-4 vs. 2.75e-4 discrepancy at #4/#9 between `old_holm_p_paper` and `old_holm_p_recomputed` in the table below) — this redo unifies everything to family=9/metric.

In [ ]:
cols = ['comparison_id', 'comparison', 'metric',
        'old_holm_p_paper', 'old_raw_p_paper', 'old_sig_paper',
        'old_wilcoxon_p_recomputed', 'old_holm_p_recomputed',
        'old_frozen_wilcoxon_p',
        'perm_p', 'holm_p', 'new_sig',
        'old_sig_effective', 'old_sig_source', 'conclusion_flipped']
tbl = old_vs_new[cols].copy()
for c in tbl.columns:
    if tbl[c].dtype == float:
        tbl[c] = tbl[c].map(lambda v: f'{v:.3g}' if pd.notna(v) else '—')
tbl

## 2. Overview of Flipped Conclusions

`conclusion_flipped` = old conclusion (the paper's explicit statement takes priority; where the paper doesn't print a value, falls back to the old-protocol recomputation) ≠ new conclusion (permutation p after Holm correction < 0.05). This field is generated programmatically from the numbers, not hand-written.

In [ ]:
flipped = robust[robust['conclusion_flipped']]
n_paper_anchored = int((robust['old_sig_source'] == 'paper').sum())
print(f'Flipped cells: {len(flipped)} / {len(robust)}')
print(f'(Of these, {n_paper_anchored} cells have old conclusions anchored to explicit paper values, '
      f'the remaining {len(robust) - n_paper_anchored} cells are anchored to the old-protocol recomputation)')
if len(flipped):
    display(flipped[['comparison_id', 'comparison', 'metric',
                     'old_sig_effective', 'old_sig_source',
                     'perm_p', 'holm_p', 'new_sig']])
else:
    print('\nAll 27 (comparison × metric) conclusions remain unchanged under the new protocol — '
          'the paper conclusions are robust to the change in statistical protocol.')

## 3. Per-Seed Permutation p Transparency (Resolving the Median-p Concern)

The permutation p-values for all 3 seeds are reported as-is, with no combining; the Cauchy-combined column is for reference only (not the primary figure — the primary figure is the permutation p after seed averaging). The earlier practice of taking the median p across three seeds is discontinued.

In [ ]:
ps = per_seed.copy()
for c in ps.columns:
    if ps[c].dtype == float:
        ps[c] = ps[c].map('{:.3g}'.format)
ps.drop(columns=['cauchy_note'])

## 4. mixedlm Robustness Branch (region + seed crossed random intercepts)

A hierarchical-model alternative: `diff ~ 1 + (1|region) + (1|seed)` (statsmodels MixedLM, REML). This applies only to the 6 comparisons involving the GNN (static-vs-static comparisons have no seed dimension). Fit status is reported as-is, including any non-converged cells.

In [ ]:
mm_rows = []
for key, v in mixed['models'].items():
    mm_rows.append({
        'comparison_id': v['comparison_id'], 'comparison': v['comparison'],
        'metric': v['metric'], 'coef': v['coef'], 'p': v['p'],
        'status': v['status'], 'var_region': v['var_region'],
        'var_seed': v['var_seed'],
        'agree_with_perm_at_0.05': v['agree_with_perm_at_0.05'],
    })
mm_df = pd.DataFrame(mm_rows)
print('Summary:', mixed['summary'])
mm_df

## 5. Moran's I Spatial Dependence Diagnostic

This is a diagnostic, not a test — no multiple-comparison correction is applied. Primary W = queen adjacency after dissolving `ITL3_region.gpkg` into the 16 study regions (row-standardized, no zero rows); supplementary W = centroid kNN(k=3).
Cells with a queen two-sided permutation p < 0.05 receive `spatial_warning`, along with a spatial-block permutation replacement p (sign flips restricted to connected blocks under queen adjacency; greedy maximum matching pairs adjacent regions into 8 pairs, each pair flipped together as a block, 2⁸=256 exhaustive enumeration, minimum attainable p = 2/2⁸ ≈ 0.0078).

In [ ]:
print('Spatial block partition (greedy matching on queen adjacency):')
for blk in morans['W_meta']['blocks_greedy_matching']:
    print('  ', blk)
mi_rows = []
for key, d in morans['diagnostics'].items():
    mi_rows.append({
        'comparison': d['comparison'], 'metric': d['metric'],
        'I_queen': d['queen']['I'], 'p_queen': d['queen']['p_perm_two_sided'],
        'I_knn3': d['knn3']['I'], 'p_knn3': d['knn3']['p_perm_two_sided'],
        'spatial_warning': d['spatial_warning'],
        'block_perm_p': d['block_perm_p'],
    })
mi_df = pd.DataFrame(mi_rows)
print(f"\nqueen-significant (flagged) cells: {int(mi_df['spatial_warning'].sum())} / {len(mi_df)}")
mi_df[mi_df['spatial_warning']]

In [ ]:
# Check whether the block-permutation replacement p for spatially flagged cells agrees with the primary test's conclusion
warn = mi_df[mi_df['spatial_warning']].copy()
warn = warn.merge(robust[['comparison', 'metric', 'perm_p', 'holm_p', 'new_sig']],
                  on=['comparison', 'metric'])
warn['block_p_sig_at_0.05'] = warn['block_perm_p'] < 0.05
warn['conclusion_robust_to_block_perm'] = warn['block_p_sig_at_0.05'] == warn['new_sig']
warn[['comparison', 'metric', 'perm_p', 'new_sig',
      'block_perm_p', 'block_p_sig_at_0.05', 'conclusion_robust_to_block_perm']]

## 6. Static-Arm Recomputation Anchoring (Handling the GPMpostP Gap)

Strong anchor: the 3 frozen arms (`voronoi_gpm` / `voronoi_ntl_gpm` / `voronoi_prox2_ntl_gpm`) match per-region within the storage precision of the frozen CSV (rmse/mae to 2 decimal places → |Δ|≤0.005; corr to 4 decimal places → |Δ|≤5×10⁻⁵).
Weak anchor: GPMpostP against the values transcribed from the paper's Table 3, ±0.005 (the paper's mean is computed as "round then average"; passing under either of the two conventions is sufficient).

In [ ]:
print('Strong anchor (all must pass to proceed):')
sa = pd.DataFrame(anchor['strong_anchors']).T
display(sa)
print('Weak anchor (GPMpostP vs. paper Table 3):')
pd.DataFrame(anchor['weak_anchor']).T

## 7. Summary of Conclusions

1. **Zero flips**: all 27 (comparison × metric) cells reach the same conclusion under the new "exact sign-flip permutation + Holm (family=9/metric)" protocol as in the paper — the core antagonism finding (#4 GNNpostNP vs. GNNpostP, RMSE +1.78, perm p = 3.05×10⁻⁵ = the minimum attainable value) and the static synergy chain (#5/#6/#7) all remain significant.
2. **Resolving the per-seed median concern**: per-seed permutation p-values are fully transparent (all three seeds are significant in the same direction for the significant cells across the six GNN comparisons; the seed-to-seed disagreement for #8 Corr is reported as-is: 0.392 / 0.018 / 0.004 — exactly the kind of case where taking a median would be inappropriate).
3. **mixedlm branch**: 17 of 18 models converged (1 case had the seed variance component land on the boundary and failed to converge, but its coefficient/p are still readable); 18/18 agree with the permutation test's conclusion at the 0.05 level.
4. **Spatial dependence**: 8/27 cells show significant queen Moran's I. Of these, 6 cells were significant in the primary test, and the block-permutation replacement p = 0.0078 (the minimum attainable value under this permutation scheme) remains < 0.05, so the conclusion is robust; the other 2 cells (#3 RMSE/MAE) were already non-significant in the primary test, and the block-permutation p ≈ 0.56 is likewise non-significant — no cell's conclusion changes due to spatial dependence.
5. **Audit findings** (to be reported in the summary): (i) the frozen static CSV is missing the GPMpostP arm (referenced in the paper's Table 3/4); this experiment recomputes it following the same definition as script 003, backed by strong/weak anchoring; (ii) the Holm p-values in the paper's Table 4 mix results from two different families (7 and 9); the value 2.1×10⁻⁴ at #4/#9 should be 2.75×10⁻⁴ (the conclusion is unaffected); (iii) the note in the paper's Table 4 stating "p is the median of three seeds" does not match the actual computation (it is actually the test after seed averaging) — this table note needs correcting during the revision.